# NLP Information Extraction

## Objective

In this notebook, we will build an NLP-based system that takes one complete job posting text and extracts useful information.

The system will extract:

- Job Title
- Company
- Location
- Salary
- Experience
- Employment Type
- Education
- Skills
- Other useful details

The extracted information will later be used by the FastAPI backend and Streamlit frontend.

## Approach

We will use:

- Regex
- spaCy
- Skill dictionary
- Rule-based extraction
- Text processing

We will not use an LLM API for the core extraction system.

## Input

One complete job posting text.

## Output

Structured job information in dictionary/JSON format.

In [82]:
import pandas as pd

In [83]:
df = pd.read_csv("../Data/fake_job_postings.csv")

In [11]:
import spacy

print("spaCy version:", spacy.__version__)

spaCy version: 3.8.16


In [12]:
nlp = spacy.load("en_core_web_sm")

print("spaCy English model loaded successfully.")

spaCy English model loaded successfully.


# Step 13B — Sample Job Posting

For testing our NLP extraction system, we will use one complete job posting as input.

The final application will also accept one complete job posting instead of asking the user to enter each field separately.

In [13]:
sample_job_posting = """
Python Developer

ABC Technologies Pvt. Ltd. is hiring a Python Developer for our Pune office.

We are looking for candidates with 1-2 years of experience in Python development.
The candidate should have a Bachelor's degree in Computer Science, Information
Technology, or a related field.

Required skills include Python, SQL, Machine Learning, FastAPI, Pandas,
NumPy, Scikit-learn, Git, and Docker.

Employment Type: Full-time

Salary: ₹4,00,000 - ₹6,00,000 per year.

Location: Pune, Maharashtra, India.

The selected candidate will work on developing and maintaining machine learning
applications and APIs. Good communication and problem-solving skills are required.
"""

print(sample_job_posting)


Python Developer

ABC Technologies Pvt. Ltd. is hiring a Python Developer for our Pune office.

We are looking for candidates with 1-2 years of experience in Python development.
The candidate should have a Bachelor's degree in Computer Science, Information
Technology, or a related field.

Required skills include Python, SQL, Machine Learning, FastAPI, Pandas,
NumPy, Scikit-learn, Git, and Docker.

Employment Type: Full-time

Salary: ₹4,00,000 - ₹6,00,000 per year.

Location: Pune, Maharashtra, India.

The selected candidate will work on developing and maintaining machine learning
applications and APIs. Good communication and problem-solving skills are required.



In [14]:
print("Input type:", type(sample_job_posting))
print("Input length:", len(sample_job_posting))

Input type: <class 'str'>
Input length: 671


In [15]:
doc = nlp(sample_job_posting)

print("Number of tokens:", len(doc))

Number of tokens: 140


In [16]:
for ent in doc.ents:
    print(ent.text, "→", ent.label_)

Python Developer

ABC Technologies Pvt. Ltd. → ORG
Pune → LOC
1-2 years → DATE
Bachelor → ORG
Computer Science, Information
Technology → ORG
SQL → ORG
Machine Learning → PERSON
Pandas → PERSON
NumPy → GPE
Git → PERSON
Docker → PERSON
₹4,00,000 - ₹ → MONEY
6,00,000 per year → DATE
India → GPE


In [17]:
def extract_job_title(text):
    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    if not lines:
        return None

    return lines[0]

In [18]:
job_title = extract_job_title(sample_job_posting)

print("Extracted Job Title:", job_title)

Extracted Job Title: Python Developer


In [19]:
test_job_2 = """
Data Scientist

XYZ Technologies is looking for a Data Scientist with 2 years of experience.
"""

print("Extracted Job Title:", extract_job_title(test_job_2))

Extracted Job Title: Data Scientist


In [20]:
print("Empty input:", extract_job_title(""))

Empty input: None


# Extract the Company Name

In [48]:
import re

def extract_company(text):
    # Process one line at a time
    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    hiring_phrases = [
        "is hiring",
        "is looking for",
        "is seeking",
        "is recruiting",
        "are hiring",
        "are looking for",
        "are seeking",
        "are recruiting"
    ]

    for line in lines:

        # Check whether this line contains a hiring phrase
        for phrase in hiring_phrases:

            if phrase.lower() not in line.lower():
                continue

            # Get the part before the hiring phrase
            before_phrase = re.split(
                re.escape(phrase),
                line,
                flags=re.IGNORECASE
            )[0].strip()

            # Remove common introductory text
            prefixes = [
                "we are",
                "we're",
                "our company",
                "company"
            ]

            for prefix in prefixes:
                if before_phrase.lower().startswith(prefix):
                    before_phrase = before_phrase[len(prefix):].strip()

            # We need an actual company-like name
            if before_phrase:

                # Remove common connecting words
                before_phrase = re.sub(
                    r"\b(our|the|team)\b\s*$",
                    "",
                    before_phrase,
                    flags=re.IGNORECASE
                ).strip()

                # Avoid returning obvious non-company phrases
                if before_phrase.lower() not in [
                    "we",
                    "we are",
                    "we're",
                    "our",
                    "company"
                ]:
                    return before_phrase

    return None

In [49]:
print("Extracted Company:", extract_company(sample_job_posting))

Extracted Company: ABC Technologies Pvt. Ltd.


In [50]:
test_job_2 = """
Data Scientist

XYZ Analytics Pvt. Ltd. is looking for a Data Scientist
with 2 years of experience in machine learning.
"""

print("Extracted Company:", extract_company(test_job_2))

Extracted Company: XYZ Analytics Pvt. Ltd.


In [51]:
test_job_3 = """
Python Developer

We are looking for a Python Developer with 2 years
of experience.
"""

print("Extracted Company:", extract_company(test_job_3))

Extracted Company: None


# Extract Location

In [70]:
import re

def extract_location(text):
    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    # 1. Check for explicit "Location:" field
    for line in lines:
        match = re.search(
            r"^\s*location\s*:\s*(.+)$",
            line,
            flags=re.IGNORECASE
        )

        if match:
            location = match.group(1).strip().rstrip(".,;")
            return location

    # 2. Check for remote location patterns
    for line in lines:
        match = re.search(
            r"\b(Remote(?:\s*[-–]\s*[A-Za-z]+)?)\b",
            line,
            flags=re.IGNORECASE
        )

        if match:
            return match.group(1).strip().rstrip(".,;")

    # 3. Use spaCy as a fallback
    doc = nlp(text)

    locations = [
        ent.text.strip().rstrip(".,;")
        for ent in doc.ents
        if ent.label_ in ["GPE", "LOC"]
    ]

    if locations:
        return ", ".join(locations)

    # 4. If no location can be confidently found
    return None

In [71]:
location = extract_location(sample_job_posting)

print("Extracted Location:", location)

Extracted Location: Pune, Maharashtra, India


In [72]:
print("Extracted Location:", extract_location(sample_job_posting))

Extracted Location: Pune, Maharashtra, India


In [73]:
test_location_2 = """
Machine Learning Engineer

ABC AI is hiring a developer for its Mumbai, Maharashtra office.
"""

print("Extracted Location:", extract_location(test_location_2))

Extracted Location: Mumbai


In [74]:
test_location_3 = """
Data Scientist

XYZ Analytics is hiring for a Remote - India position.
"""

print("Extracted Location:", extract_location(test_location_3))

Extracted Location: Remote - India


# Extract Salary

In [97]:
import re

def extract_salary(text):
    patterns = [
        # Currency + range
        r'(?:(?:₹|Rs\.?|INR|\$|USD|€|EUR|£|GBP)\s*)'
        r'[\d,]+(?:\.\d+)?\s*(?:-|–|to)\s*'
        r'(?:(?:₹|Rs\.?|INR|\$|USD|€|EUR|£|GBP)\s*)?'
        r'[\d,]+(?:\.\d+)?'
        r'(?:\s*(?:per|a|/)\s*(?:year|month|annum|hour|yr|mo))?',

        # Number + LPA / lakh / lakhs
        r'[\d,.]+\s*(?:-\s*[\d,.]+\s*)?'
        r'(?:LPA|lakhs?|lakh)\b'
        r'(?:\s*(?:per|a)\s*(?:year|annum))?',

        # Currency + k/K salary range
        r'(?:(?:₹|Rs\.?|INR|\$|USD|€|EUR|£|GBP)\s*)'
        r'\d+(?:\.\d+)?\s*[Kk]\s*(?:-|–|to)\s*'
        r'(?:(?:₹|Rs\.?|INR|\$|USD|€|EUR|£|GBP)\s*)?'
        r'\d+(?:\.\d+)?\s*[Kk]',

        # Currency + single salary
        r'(?:(?:₹|Rs\.?|INR|\$|USD|€|EUR|£|GBP)\s*)'
        r'[\d,]+(?:\.\d+)?'
        r'(?:\s*(?:per|a|/)\s*(?:year|month|annum|hour|yr|mo))?',

        # Number range with k/K suffix
        r'\b\d+(?:\.\d+)?\s*[Kk]\s*(?:-|–|to)\s*'
        r'\d+(?:\.\d+)?\s*[Kk]\b',

        # Number range without currency
        r'\b\d+(?:,\d{3})*(?:\.\d+)?\s*(?:-|–|to)\s*'
        r'\d+(?:,\d{3})*(?:\.\d+)?'
        r'(?:\s*(?:per|a|/)\s*(?:year|month|annum|hour|yr|mo))?'
    ]

    for pattern in patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)

        if match:
            return match.group(0).strip()

    return None

In [98]:
salary = extract_salary(sample_job_posting)

print("Extracted Salary:", salary)

Extracted Salary: ₹4,00,000 - ₹6,00,000 per year


In [99]:
test_salary_1 = "The salary offered is 4-6 LPA."

print(extract_salary(test_salary_1))

4-6 LPA


In [100]:
test_salary_3 = "The expected salary is 40k-60k."

print(extract_salary(test_salary_3))

40k-60k


In [101]:
test_salary_4 = "We offer a competitive salary based on experience."

print(extract_salary(test_salary_4))

None


In [102]:
test_salaries = [
    "Salary: ₹4,00,000 - ₹6,00,000 per year.",
    "The salary offered is 4-6 LPA.",
    "Salary: $80,000 - $100,000 per year.",
    "The expected salary is 40k-60k.",
    "We offer a competitive salary based on experience.",

    # Unseen formats
    "The salary is ₹75,000/month.",
    "CTC: ₹7.5 LPA.",
    "Compensation is €40,000 annually.",
    "Pay range is £35k - £45k.",
    "Expected CTC is 5 lakhs per annum."
]

for text in test_salaries:
    print("Input:", text)
    print("Extracted:", extract_salary(text))
    print("-" * 60)

Input: Salary: ₹4,00,000 - ₹6,00,000 per year.
Extracted: ₹4,00,000 - ₹6,00,000 per year
------------------------------------------------------------
Input: The salary offered is 4-6 LPA.
Extracted: 4-6 LPA
------------------------------------------------------------
Input: Salary: $80,000 - $100,000 per year.
Extracted: $80,000 - $100,000 per year
------------------------------------------------------------
Input: The expected salary is 40k-60k.
Extracted: 40k-60k
------------------------------------------------------------
Input: We offer a competitive salary based on experience.
Extracted: None
------------------------------------------------------------
Input: The salary is ₹75,000/month.
Extracted: ₹75,000/month
------------------------------------------------------------
Input: CTC: ₹7.5 LPA.
Extracted: 7.5 LPA
------------------------------------------------------------
Input: Compensation is €40,000 annually.
Extracted: €40,000
--------------------------------------------------

# Extract Experience

In [106]:
import re


def extract_experience(text):
    """
    Extract experience-related information from job posting text.
    Removes overlapping matches and returns unique mentions.
    """

    patterns = [
        # Numeric ranges
        r'\b\d+(?:\.\d+)?\s*(?:-|–|—|to)\s*'
        r'\d+(?:\.\d+)?\s*(?:years?|yrs?|months?)\b',

        # Minimum: 2+ years
        r'\b\d+(?:\.\d+)?\s*\+\s*(?:years?|yrs?|months?)\b',

        # Minimum wording
        r'\b(?:minimum|min\.?|at\s+least)\s+'
        r'(?:of\s+)?\d+(?:\.\d+)?\s*'
        r'(?:years?|yrs?|months?)\b',

        # Maximum wording
        r'\b(?:up\s+to|less\s+than|under)\s+'
        r'\d+(?:\.\d+)?\s*'
        r'(?:years?|yrs?|months?)\b',

        # 3 years or more
        r'\b\d+(?:\.\d+)?\s*(?:years?|yrs?|months?)'
        r'\s+or\s+more\b',

        # Single experience requirement
        r'\b\d+(?:\.\d+)?\s*(?:years?|yrs?|months?)'
        r'(?:\s+of)?\s+'
        r'(?:(?:relevant|professional|industry|work|prior|'
        r'hands[-\s]?on)\s+)*experience\b',

        # experience of 3 years
        r'\bexperience\s+of\s+\d+(?:\.\d+)?\s*'
        r'(?:years?|yrs?|months?)\b',

        # Fresher
        r'\bfreshers?\b',
        r'\bfresh\s+graduates?\b',
        r'\brecent\s+graduates?\b',
        r'\bnew\s+graduates?\b',

        # No experience
        r'\bno\s+(?:prior\s+|previous\s+|professional\s+)?'
        r'experience\s+(?:is\s+)?required\b',

        r'\bexperience\s+(?:is\s+)?not\s+required\b',

        # Experience levels
        r'\bentry[-\s]?level\b',
        r'\bjunior(?:[-\s]?level)?\b',
        r'\bmid[-\s]?level\b',
        r'\bintermediate[-\s]?level\b',
        r'\bsenior(?:[-\s]?level)?\b',
        r'\bexperienced\s+professionals?\b'
    ]

    candidates = []

    for pattern in patterns:
        for match in re.finditer(pattern, text, flags=re.IGNORECASE):
            candidates.append({
                "text": match.group(0).strip(),
                "start": match.start(),
                "end": match.end()
            })

    # Sort by starting position, then prefer longer matches
    candidates.sort(
        key=lambda x: (x["start"], -(x["end"] - x["start"]))
    )

    selected = []

    for candidate in candidates:
        overlap = False

        for existing in selected:
            if (
                candidate["start"] < existing["end"]
                and candidate["end"] > existing["start"]
            ):
                overlap = True
                break

        if not overlap:
            selected.append(candidate)

    # Remove duplicates while preserving order
    results = []
    seen = set()

    for item in sorted(selected, key=lambda x: x["start"]):
        value = item["text"]

        if value.lower() not in seen:
            results.append(value)
            seen.add(value.lower())

    return results if results else None

In [107]:
test_text = """
We are hiring a Machine Learning Engineer.

Candidates should have 1.5-3.5 years of hands-on experience.
A minimum of 18 months of Python development experience is preferred.
Senior candidates with 7+ years experience may also apply.

Fresh graduates with strong projects are welcome for our trainee role.
"""

print(extract_experience(test_text))

['1.5-3.5 years', 'minimum of 18 months', 'Senior', '7+ years', 'Fresh graduates']


In [108]:
print(
    extract_experience(
        "We are hiring a Python developer. "
        "Strong communication and SQL skills are required."
    )
)

None


# Extract Education

In [115]:
import re


def extract_education(text):
    """
    Extract education-related information from a job posting.
    Returns unique education mentions.
    """

    patterns = [
        # Bachelor's / Master's / Associate / Doctorate
        (
            r"\b(?:bachelor(?:'s)?|master(?:'s)?|associate(?:'s)?|"
            r"doctorate|doctoral)\s+(?:degree|qualification)\b",
            re.IGNORECASE
        ),

        # Degree / diploma in subject
        (
            r"\b(?:degree|diploma)\s+in\s+"
            r"[A-Za-z][A-Za-z\s,&/-]*?"
            r"(?=\s*(?:[.,;]|and|or|preferred|required|"
            r"may|can|is|are|$))",
            re.IGNORECASE
        ),

        # Degree abbreviations — CASE SENSITIVE
        (
            r"(?<![A-Za-z])"
            r"(?:B\.Tech|B\.E\.|M\.Tech|M\.E\.|B\.Sc|M\.Sc|BCA|MCA)"
            r"(?![A-Za-z])",
            0
        ),

        # MBA / PhD — also case-sensitive
        (
            r"(?<![A-Za-z])(?:MBA|Ph\.D\.)(?![A-Za-z])",
            0
        ),

        # Diploma
        (
            r"\b(?:advanced\s+)?diploma\b",
            re.IGNORECASE
        ),

        # High school
        (
            r"\bhigh\s+school(?:\s+diploma)?\b",
            re.IGNORECASE
        ),

        # Any degree
        (
            r"\bany\s+(?:bachelor(?:'s)?|master(?:'s)?|college)\s+degree\b",
            re.IGNORECASE
        ),

        # No degree required
        (
            r"\bno\s+(?:degree|formal\s+education)\s+(?:is\s+)?required\b",
            re.IGNORECASE
        )
    ]

    candidates = []

    for pattern, flags in patterns:
        for match in re.finditer(pattern, text, flags=flags):
            candidates.append({
                "text": match.group(0).strip(),
                "start": match.start(),
                "end": match.end()
            })

    # Prefer longer matches when patterns overlap
    candidates.sort(
        key=lambda x: (
            x["start"],
            -(x["end"] - x["start"])
        )
    )

    selected = []

    for candidate in candidates:
        overlap = False

        for existing in selected:
            if (
                candidate["start"] < existing["end"]
                and candidate["end"] > existing["start"]
            ):
                overlap = True
                break

        if not overlap:
            selected.append(candidate)

    # Remove duplicates
    results = []
    seen = set()

    for item in sorted(selected, key=lambda x: x["start"]):
        value = item["text"].strip()

        if value.lower() not in seen:
            results.append(value)
            seen.add(value.lower())

    return results if results else None

In [116]:
education_text = """
Python Developer

The candidate should have a Bachelor's degree in Computer Science,
Information Technology, or a related field.

A B.Tech, B.E., M.Tech, or M.E. qualification is preferred.
MBA candidates may also be considered.

Candidates with a diploma in software engineering can apply.
Fresh graduates with no formal education are not eligible.
"""

print(extract_education(education_text))

["Bachelor's degree", 'B.Tech', 'B.E.', 'M.Tech', 'M.E.', 'MBA', 'diploma in softw']


# Employment Type Extraction

In [117]:
import re

def extract_employment_type(text):
    patterns = [
        r'\bfull[-\s]?time\b',
        r'\bpart[-\s]?time\b',
        r'\bcontract(?:[-\s]based)?\b',
        r'\btemporary\b',
        r'\bpermanent\b',
        r'\binternships?\b',
        r'\bfreelance\b',
        r'\bseasonal\b',
        r'\bapprenticeships?\b',
        r'\bvolunteer(?:ing)?\b',
        r'\bcasual\s+(?:employment|work|position|role)\b',
        r'\bfixed[-\s]?term\b'
    ]

    matches = []

    for pattern in patterns:
        for match in re.finditer(pattern, text, flags=re.IGNORECASE):
            value = match.group(0).strip()

            if value.lower() not in {
                item.lower() for item in matches
            }:
                matches.append(value)

    return matches if matches else None

In [118]:
employment_text = """
We are hiring a Machine Learning Engineer for a full-time position.

This is a permanent role based in Pune.
We also offer part-time and freelance opportunities.

Students can apply for our internship program.
Some positions are available on a fixed-term contract.
"""

print(extract_employment_type(employment_text))

['full-time', 'part-time', 'contract', 'permanent', 'internship', 'freelance', 'fixed-term']


In [119]:
print(
    extract_employment_type(
        "We are looking for a Python developer with strong SQL skills."
    )
)

None


# Skill Extractor

In [120]:
import sys

sys.path.append("../src")

from nlp.skill_dictionary import SKILL_ALIASES

In [121]:
import re


def extract_skills(text, skill_aliases=SKILL_ALIASES):
    """
    Extract canonical skills from job-posting text.

    Different aliases of the same skill are normalized
    to one canonical skill name.
    """

    found_skills = []

    for canonical_skill, aliases in skill_aliases.items():

        for alias in aliases:

            # Escape the alias so characters like +, #, . etc.
            # are treated literally.
            pattern = r"(?<!\w)" + re.escape(alias) + r"(?!\w)"

            if re.search(pattern, text, flags=re.IGNORECASE):
                found_skills.append(canonical_skill)
                break

    return found_skills if found_skills else None

In [122]:
skill_test = """
We are looking for an ML Engineer with experience in Python,
sklearn, NLP, PyTorch, FastAPI, PostgreSQL, Docker and k8s.

Knowledge of REST APIs and problem solving skills is required.
Good communication and teamwork are also important.
"""

print(extract_skills(skill_test))

['Python', 'Machine Learning', 'PyTorch', 'Natural Language Processing', 'Scikit-learn', 'FastAPI', 'REST API', 'PostgreSQL', 'Docker', 'Kubernetes', 'REST', 'Communication', 'Teamwork', 'Problem Solving']


In [123]:
skill_test_2 = """
PYTHON, machine learning, SKLEARN, nLp, Docker,
POSTGRES and KUBERNETES are required.
"""

print(extract_skills(skill_test_2))

['Python', 'Machine Learning', 'Natural Language Processing', 'Scikit-learn', 'PostgreSQL', 'Docker', 'Kubernetes']


In [124]:
print(
    extract_skills(
        "We are looking for a candidate who can join our office next Monday."
    )
)

None


In [125]:
ambiguous_skill_test = """
We are looking for a data scientist.

The candidate should have strong Python, SQL and R programming
experience. Knowledge of C++ and Go is preferred.

Experience with AI and ML projects is required.
"""

print(extract_skills(ambiguous_skill_test))

['Python', 'C++', 'R', 'SQL', 'Machine Learning']


In [126]:
false_positive_test = """
We are hiring a software engineer for our Pune office.

The candidate should have strong communication skills and work
closely with our team. Good problem solving and analytical thinking
are required.
"""

print(extract_skills(false_positive_test))

['Communication', 'Problem Solving', 'Analytical Thinking']


In [127]:
print("Go" in SKILL_ALIASES)
print("AI" in SKILL_ALIASES)
print("Machine Learning" in SKILL_ALIASES)

print(SKILL_ALIASES.get("Go"))
print(SKILL_ALIASES.get("AI"))

True
False
True
['golang', 'go language']
None
